<a href="https://colab.research.google.com/github/Gobishangar1/CDAZZDEV-MLE-GobishangarVijayendran/blob/main/task3_agentic/notebooks/task3_agentic_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gobishangar1/CDAZZDEV-MLE-GobishangarVijayendran/blob/main/task3_agentic/notebooks/task3_agentic_research.ipynb)

**Candidate:** Gobishangar Vijayendran · **Ticker:** NVDA · **Framework:** LangGraph · **LLM:** Groq `openai/gpt-oss-120b`
(fallback `openai/gpt-oss-20b`)

The query (from the brief):
> Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share
> price over the next 90 days and suggest one data-driven hedge strategy.

| Rubric part | Section |
|---|---|
| 3A All five tools implemented (types) | 2 |
| 3A Autonomous tool selection, observe-and-replan cycle, final report | 3 |
| 3A Error handling (tool failure, agent replans) | 4 |
| 3C Short-term memory (follow-up with no tool call) | 5 |
| 3B Roles and tool restriction | 6 |
| 3B Structured handoff, message trace, critique loop, end-to-end | 7 |
| 3C Persistent memory (JSON cache by ticker and date) | 8 |
| 3C Observability (`logs/agent_trace.jsonl`) + bonus dashboard | 9 |

All logic is in the `research_agents` package. This notebook only calls it and shows the results.

## How it works

```
3A  one agent, all five tools (LangGraph ReAct loop)

    START -> agent --(tool call?)--> tools -> agent -> ... --(no tool call)--> END -> report writer (JSON, Pydantic)
             ^ the LLM decides the next tool from what it has seen; Python only runs the tool and returns the result

3B  two agents, restricted tools, typed handoffs, one critique round

    Agent A (Data Analyst)    get_price_data, calculate_volatility, llm_sentiment   (no news, no web)
    Agent B (Research Writer) web_search, get_news                                  (no price tools)

    analyst --DataBrief--> writer_research -> writer_critique --ClarificationRequest--> analyst_clarify
            --ClarificationResponse--> writer_report -> ResearchReport (must say how the answer was used)
```

Why they need each other: Agent A can score sentiment but cannot fetch news. Agent B has the news but cannot score it,
and has no price or volatility data. So B sends its headlines to A in the clarification request.

## 0. Setup

In [1]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Task 3 notebook cells in rubric order', Date: 2026-10-11
import os, sys

REPO = "CDAZZDEV-MLE-GobishangarVijayendran"
if "google.colab" in sys.modules:                        # Colab: clone the repo and install
    if not os.path.exists(f"/content/{REPO}"):
        !git clone -q https://github.com/Gobishangar1/{REPO}.git /content/{REPO}
    %cd /content/{REPO}/task3_agentic
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":        # local Jupyter started inside notebooks/
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

/content/CDAZZDEV-MLE-GobishangarVijayendran/task3_agentic
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 27.2 MB/s eta 0:00:00
Working directory: /content/CDAZZDEV-MLE-GobishangarVijayendran/task3_agentic


In [2]:
# The Groq key comes from Colab Secrets (key icon on the left) or the environment. It is never printed.
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

from research_agents.config import (DEFAULT_TICKER, RESEARCH_QUERY, TRACE_FILE, load_colab_secret, require_groq_key,
                                    setup_logging)

loaded = load_colab_secret("GROQ_API_KEY")
require_groq_key()
print("GROQ_API_KEY available: True | loaded from Colab Secrets:", loaded)
setup_logging()
pd.set_option("display.max_colwidth", 140)

TICKER = DEFAULT_TICKER
QUERY = RESEARCH_QUERY.format(ticker=TICKER)
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")        # run ids in the trace file
print("Query:", QUERY)

GROQ_API_KEY available: True | loaded from Colab Secrets: True
Query: Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.


In [3]:
from research_agents.llm import make_agent_llm, make_json_llm, make_sentiment_llm
from research_agents.tools import ALL_TOOLS, ToolBox
from research_agents.trace import Tracer

agent_llm = make_agent_llm()          # tool calling: gpt-oss-120b, fallback gpt-oss-20b
json_llm = make_json_llm()            # structured outputs (report, clarification), JSON mode
sentiment_llm = make_sentiment_llm()  # the llm_sentiment tool: gpt-oss-20b, JSON mode
print("Models ready. Trace file:", TRACE_FILE)

Models ready. Trace file: /content/CDAZZDEV-MLE-GobishangarVijayendran/task3_agentic/logs/agent_trace.jsonl


## 1. Configuration
Every limit is a named constant (step budget, output caps, windows). Tool outputs sent to the LLM are capped to keep
the free-tier token budget (about 8K tokens per minute per model).

In [4]:
import research_agents.config as cfg
names = ["AGENT_MODEL", "FALLBACK_MODEL", "SENTIMENT_MODEL", "AGENT_MAX_STEPS", "TOOL_OUTPUT_TO_LLM_CHARS",
         "TRACE_OUTPUT_CHARS", "DEFAULT_VOL_WINDOW", "TRADING_DAYS_PER_YEAR", "MAX_SENTIMENT_HEADLINES",
         "CLARIFICATION_ROUNDS", "JSON_ATTEMPTS"]
pd.DataFrame([(n, getattr(cfg, n)) for n in names], columns=["constant", "value"])

,constant,value
0,AGENT_MODEL,openai/gpt-oss-120b
1,FALLBACK_MODEL,openai/gpt-oss-20b
2,SENTIMENT_MODEL,openai/gpt-oss-20b
3,AGENT_MAX_STEPS,8
4,TOOL_OUTPUT_TO_LLM_CHARS,2500
5,TRACE_OUTPUT_CHARS,200
6,DEFAULT_VOL_WINDOW,30
7,TRADING_DAYS_PER_YEAR,252
8,MAX_SENTIMENT_HEADLINES,15
9,CLARIFICATION_ROUNDS,1


## 2. Task 3A: the five tools
Each tool has a Pydantic argument schema (what the LLM sees) and returns a Pydantic model. Here each tool is run
through `ToolBox.execute()`, the same path the agents use, so every call is also written to `agent_trace.jsonl`.

In [5]:
tracer_tools = Tracer(run_id=f"tools-{STAMP}")
box = ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_tools)

print("Tool schemas the LLM sees:")
for t in box.langchain_tools(list(ALL_TOOLS)):
    print(f"  {t.name}({', '.join(t.args)})")

results, rows = {}, []
calls = [("get_price_data", {"ticker": TICKER, "period": "1y"}),
         ("get_news", {"ticker": TICKER, "n": 10}),
         ("calculate_volatility", {"ticker": TICKER, "window": 30}),
         ("web_search", {"query": f"{TICKER} stock analyst outlook risks", "max_results": 5})]
for name, args in calls:
    text, status, result = box.execute("notebook", set(ALL_TOOLS), name, args)
    results[name] = result
    rows.append({"tool": name, "args": args, "status": status, "returns": type(result).__name__, "preview": text[:150]})

headlines = [h.title for h in results["get_news"].headlines] if results["get_news"] else ["NVIDIA shares rise"]
text, status, result = box.execute("notebook", set(ALL_TOOLS), "llm_sentiment", {"headlines": headlines[:10]})
results["llm_sentiment"] = result
rows.append({"tool": "llm_sentiment", "args": {"headlines": f"{len(headlines[:10])} headlines"}, "status": status,
             "returns": type(result).__name__, "preview": text[:150]})
pd.DataFrame(rows)

Tool schemas the LLM sees:
  get_price_data(ticker, period)
  get_news(ticker, n)
  calculate_volatility(ticker, window)
  llm_sentiment(headlines)
  web_search(query, max_results)
      <- get_price_data [OK, 1363 ms]: {"ticker":"NVDA","period":"1y","start":"2025-10-09","end":"2026-10-09","rows":252,"last_close":229.28,"period_return_pct":19.35,"high_52w":243.37,"low_52w":163.9,"pct_from_52w_high":-5.79,"indicators":{"sma_50":221.9403,"sma_200":201.8289,"rsi_14":53.0274,"macd":3.9885,"macd_signal":3.87,"macd_hist":0.1185,"bb_upper":242.4239,"bb_middle":226.734,"bb_lower":211.0441,"bb_pct_b":0.5811},"trend":"c...
      <- get_news [OK, 3447 ms]: {"ticker":"NVDA","headlines":[{"title":"Meet the Only AI Stock I'd Buy Over Nvidia in October","publisher":"fool.com","published_at":"2026-10-11T10:37:00+00:00","source":"yahoo_rss","relevant":true},{"title":"Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Corporation $NVDA - MarketBeat","publisher":"MarketBeat","published_at":"2

,tool,args,status,returns,preview
0,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}",ok,PriceData,"{""ticker"":""NVDA"",""period"":""1y"",""start"":""2025-10-09"",""end"":""2026-10-09"",""rows"":252,""last_close"":229.28,""period_return_pct"":19.35,""high_52..."
1,get_news,"{'ticker': 'NVDA', 'n': 10}",ok,NewsResult,"{""ticker"":""NVDA"",""headlines"":[{""title"":""Meet the Only AI Stock I'd Buy Over Nvidia in October"",""publisher"":""fool.com"",""published_at"":""20..."
2,calculate_volatility,"{'ticker': 'NVDA', 'window': 30}",ok,VolatilityResult,"{""ticker"":""NVDA"",""window"":30,""as_of"":""2026-10-09"",""annualised_vol_pct"":29.45,""one_year_vol_pct"":37.66,""vol_percentile_1y"":6.3,""max_drawd..."
3,web_search,"{'query': 'NVDA stock analyst outlook risks', 'max_results': 5}",ok,SearchResult,"{""query"":""NVDA stock analyst outlook risks"",""results"":[{""title"":""NVDA Stock Analysis & Analyst Ratings | VCP Scanner"",""url"":""https://vcp..."
4,llm_sentiment,{'headlines': '10 headlines'},ok,SentimentResult,"{""n"":10,""score"":0.0,""label"":""neutral"",""counts"":{""positive"":5,""neutral"":0,""negative"":5},""items"":[{""headline"":""Meet the Only AI Stock I'd ..."


In [6]:
p, v = results["get_price_data"], results["calculate_volatility"]
if p:
    print(f"Price: last close {p.last_close} | {p.period} return {p.period_return_pct}% | "
          f"{p.pct_from_52w_high}% from 52-week high\nTrend (computed by code): {p.trend}")
    display(pd.DataFrame([p.indicators]))
if v:
    print("\nVolatility:", v.interpretation)

Price: last close 229.28 | 1y return 19.35% | -5.79% from 52-week high
Trend (computed by code): close above SMA-50; close above SMA-200; long-term uptrend (SMA-50 > SMA-200); RSI 53.0 (neutral); MACD histogram positive


,sma_50,sma_200,rsi_14,macd,macd_signal,macd_hist,bb_upper,bb_middle,bb_lower,bb_pct_b
0,221.9403,201.8289,53.0274,3.9885,3.87,0.1185,242.4239,226.734,211.0441,0.5811



Volatility: 30-day annualised volatility is 29.5%, at the 6th percentile of the past year (low). On 5% of days in the past year the stock fell more than 3.9%; the worst peak-to-trough fall was 20.2%.


In [7]:
if results["get_news"]:
    display(pd.DataFrame([h.model_dump() for h in results["get_news"].headlines])[["title", "publisher", "published_at", "source"]])
if results["llm_sentiment"]:
    s = results["llm_sentiment"]
    print(f"Sentiment score {s.score:+.3f} ({s.label}), counts {s.counts}")
    display(pd.DataFrame([i.model_dump() for i in s.items]))
if results["web_search"]:
    display(pd.DataFrame([h.model_dump() for h in results["web_search"].results]))

,title,publisher,published_at,source
0,Meet the Only AI Stock I'd Buy Over Nvidia in October,fool.com,2026-10-11T10:37:00+00:00,yahoo_rss
1,"Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Corporation $NVDA - MarketBeat",MarketBeat,2026-10-11T10:08:14+00:00,google_news
2,"Palisade Capital Management LP Acquires 28,724 Shares of NVIDIA Corporation $NVDA - MarketBeat",MarketBeat,2026-10-11T10:08:13+00:00,google_news
3,NVIDIA Corporation $NVDA Stock Sold by Primecap Management Co. CA - MarketBeat,MarketBeat,2026-10-11T09:58:11+00:00,google_news
4,"Jones Financial Companies Lllp Acquires 3,087,398 Shares of NVIDIA Corporation $NVDA - MarketBeat",MarketBeat,2026-10-11T09:58:11+00:00,google_news
5,"268,305 NVIDIA Corporation $NVDA Shares Sold by The Manufacturers Life Insurance Company - MarketBeat",MarketBeat,2026-10-11T09:58:09+00:00,google_news
6,NVIDIA Corporation $NVDA Stock Sold by Kinetic Partners Management LP - MarketBeat,MarketBeat,2026-10-11T09:58:07+00:00,google_news
7,NVIDIA Corporation $NVDA Shares Purchased by Nicholas Investment Partners LP - MarketBeat,MarketBeat,2026-10-11T09:58:07+00:00,google_news
8,NVIDIA Corporation $NVDA Stake Raised by Wedbush Fund Advisers LLC - MarketBeat,MarketBeat,2026-10-11T09:58:07+00:00,google_news
9,NVIDIA Corporation $NVDA Shares Sold by Legal & General Group Plc - MarketBeat,MarketBeat,2026-10-11T09:58:02+00:00,google_news


Sentiment score +0.000 (neutral), counts {'positive': 5, 'neutral': 0, 'negative': 5}


,headline,sentiment,confidence,brief_reason
0,Meet the Only AI Stock I'd Buy Over Nvidia in October,positive,0.95,"Analyst recommends buying Nvidia over competitors, indicating bullish sentiment."
1,"Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Corporation $NVDA - MarketBeat",negative,0.95,Institutional sell indicates potential downside pressure on Nvidia shares.
2,"Palisade Capital Management LP Acquires 28,724 Shares of NVIDIA Corporation $NVDA - MarketBeat",positive,0.95,Large institutional purchase signals confidence in Nvidia's prospects.
3,NVIDIA Corporation $NVDA Stock Sold by Primecap Management Co. CA - MarketBeat,negative,0.95,Sell order by a fund suggests possible negative outlook.
4,"Jones Financial Companies Lllp Acquires 3,087,398 Shares of NVIDIA Corporation $NVDA - MarketBeat",positive,0.95,Massive buy by a financial firm indicates strong bullish sentiment.
5,"268,305 NVIDIA Corporation $NVDA Shares Sold by The Manufacturers Life Insurance Company - MarketBeat",negative,0.95,Large sell by an insurer may signal concern about Nvidia.
6,NVIDIA Corporation $NVDA Stock Sold by Kinetic Partners Management LP - MarketBeat,negative,0.95,Sell order from a partner fund suggests potential downside.
7,NVIDIA Corporation $NVDA Shares Purchased by Nicholas Investment Partners LP - MarketBeat,positive,0.95,Purchase by an investment partner indicates bullish view.
8,NVIDIA Corporation $NVDA Stake Raised by Wedbush Fund Advisers LLC - MarketBeat,positive,0.95,Raising stake shows increased confidence in Nvidia.
9,NVIDIA Corporation $NVDA Shares Sold by Legal & General Group Plc - MarketBeat,negative,0.95,Sell by a major insurer may weigh on Nvidia's share price.


,title,url,snippet
0,NVDA Stock Analysis & Analyst Ratings | VCP Scanner,https://vcpscanner.com/stock/nvda/analysis,"NVIDIA Corporation (NVDA) stock analysis: $339 consensus target (+48% upside) — Buy rating from 79 analysts. View bull case, risk factor..."
1,NVIDIA (NVDA) Stock Forecast & Analyst Price Targets,https://stockanalysis.com/stocks/nvda/forecast/,"1 day ago · Stock forecasts and analyst price target predictions for NVIDIA Corporation (NVDA) stock, with detailed revenue and earnings..."
2,"NVIDIA (NVDA) Stock Forecast and Price Target 2026Nvidia (NVDA) Stock: 2026 Outlook, Risks, Technicals & Trade ...NVIDIA (NVDA) Stock An...",https://www.marketbeat.com/stocks/NASDAQ/NVDA/forecast/,2 days ago · NVDA's current price target is $321.89. Learn why top analysts are making this stock forecast for NVIDIA at MarketBeat. Dec...
3,"NVIDIA (NVDA) Stock Analysis: Growth, Moat, Risks",https://stockmetriclab.com/nvidia-stock-analysis/,"Sep 11, 2026 · NVIDIA Corporation (NASDAQ: NVDA) has become the central supplier of accelerated computing infrastructure for artificial ..."
4,NVIDIA (NVDA) Analyst Ratings - Stock Analysis,https://stockanalysis.com/stocks/nvda/ratings/,"1 day ago · Current and historical analyst ratings for NVIDIA (NVDA) stock. See upgrades, downgrades, price targets and more from profes..."


## 3. Task 3A: one research agent, all five tools
The agent gets only the query. Nothing in the code fixes the order of tools: each turn, the LLM reads the results
so far and decides the next tool and its arguments (or decides it is done). The trace below shows the cycle:
`thinks` (the model's reasoning) -> `calls` (tool and arguments) -> `<-` (the observed result) -> next decision.

In [8]:
from research_agents.report import report_to_markdown
from research_agents.single_agent import ResearchSession

tracer_3a = Tracer(run_id=f"3A-{STAMP}")
session = ResearchSession(TICKER, agent_llm, json_llm, ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_3a),
                          tracer_3a)
result = session.research()


-------- 3A Research Agent: Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy. --------
  [task] Research Agent <- Analyse the current financial health and market sentiment of NVDA. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
  [Research Agent] thinks: We need to analyze NVDA (NVIDIA). Need price data, volatility, news, sentiment, web search for analyst commentary and risks. Then summarize financial health (price trend, indicators, volatility), market sentiment (news sentiment), top three risks with evidence, and a hedge idea with a number from tools. We'll start by getting price data for NVDA, maybe 3 months to see trend and indicators.
  [Research Agent] -> calls get_price_data({"period":"3mo","ticker":"NVDA"})
      <- get_price_data [OK, 9 ms]: {"ticker":"NVDA","period":"3mo","start":

### 3.1 The decision path (observe -> decide -> act)
Each row is one decision by the LLM. The `reasoning` column is what it wrote before choosing the tool.

In [9]:
from langchain_core.messages import AIMessage, ToolMessage

def decision_table(messages):
    rows, step = [], 0
    status = {m.tool_call_id: (m.artifact or {}).get("status") for m in messages if isinstance(m, ToolMessage)}
    for m in messages:
        if isinstance(m, AIMessage):
            step += 1
            thought = (m.content or m.additional_kwargs.get("reasoning_content", "") or "").strip()
            for c in m.tool_calls or [{"name": "(final answer)", "args": {}, "id": None}]:
                rows.append({"step": step, "reasoning": " ".join(thought.split())[:220], "tool": c["name"],
                             "args": c["args"], "result": status.get(c["id"], "")})
    return pd.DataFrame(rows)

decision_table(result.run.messages)

,step,reasoning,tool,args,result
0,1,"We need to analyze NVDA (NVIDIA). Need price data, volatility, news, sentiment, web search for analyst commentary and risks. Then summar...",get_price_data,"{'period': '3mo', 'ticker': 'NVDA'}",ok
1,2,"We have price data: 3-month period, price up 13.19%, above SMA-50 and SMA-200, RSI neutral, MACD positive, near 5.8% below 52-week high....",calculate_volatility,"{'ticker': 'NVDA', 'window': 30}",ok
2,3,"We have price data showing uptrend, price near 5.8% below 52-week high, SMA 50 above SMA 200, RSI neutral, MACD positive. Volatility low...",get_news,"{'n': 15, 'ticker': 'NVDA'}",ok
3,4,"We have price data, volatility, news headlines (mostly market beat trades). Need sentiment analysis of headlines. Also need external ana...",llm_sentiment,"{'headlines': ['Meet the Only AI Stock I'd Buy Over Nvidia in October', 'Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA ...",ok
4,5,"We have price data, volatility, news headlines, sentiment. Need to identify top three risks over next 90 days and suggest hedge with dat...",web_search,"{'max_results': 5, 'query': 'NVDA upcoming earnings date 2026 risk factors AI chip competition AMD Intel supply chain'}",ok
5,6,"We have price data showing uptrend, RSI neutral, volatility low (6th percentile). News sentiment slightly positive overall. Need to iden...",web_search,"{'max_results': 5, 'query': 'AMD Intel competition NVIDIA AI chip market share 2026 risk'}",error
6,7,"We need to identify top three risks for NVDA over next 90 days, summarize financial health, market sentiment, and suggest hedge with dat...",web_search,"{'max_results': 5, 'query': 'NVIDIA earnings November 2026 date'}",error
7,8,"We have price data, volatility, news headlines with sentiment, and some web search results about earnings date and risk factors. Need to...",web_search,"{'max_results': 5, 'query': 'NVIDIA supply chain risk'}",ok
8,9,**Financial health & market sentiment (as of 2026‑10‑09)** | Metric | Value | Interpretation | |--------|-------|----------------| | 3‑m...,(final answer),{},


### 3.2 Final report (three sections, validated with Pydantic)

In [10]:
if result.report:
    display(Markdown(report_to_markdown(result.report)))
else:
    print("Report failed:", result.error)

# NVDA research brief (2026-10-11)

## 1. Financial Health Summary

NVDA has rallied +13.19% over the past three months and is trading comfortably above both its 50‑day (221.94) and 200‑day (201.83) moving averages, indicating a sustained uptrend. Technical momentum remains positive with RSI at 53 (neutral) and a modestly positive MACD histogram (+0.12). Volatility is low for the year – 30‑day annualised volatility is 29.45%, placing it in the 6th percentile of the past 12 months, while the 1‑day 95 % VaR is –3.88%.

- 3‑month return: +13.19%
- Last close: $229.28
- SMA‑50: $221.94
- SMA‑200: $201.83
- RSI‑14: 53.0
- MACD histogram: 0.1185
- Bollinger %B: 0.5811
- 30‑day annualised volatility: 29.45%
- 1‑day 95 % VaR: –3.88%
- 1‑year max drawdown: –20.21%

**Market sentiment:** LLM sentiment analysis of recent headlines yields a modestly positive score (+0.192) with 7 bullish versus 5 bearish mentions, reflecting overall optimism tempered by visible institutional selling.

## 2. Top Three Risks (next 90 days)

### 1. Earnings surprise risk (high)

NVDA’s next earnings report is slated for 26 Aug 2026 (Q2 FY27) with analysts forecasting $93‑$95 B revenue; a miss or weaker guidance could trigger a sharp sell‑off in the next 90 days.

Evidence:
- Earnings date: 26 Aug 2026
- Revenue forecast $93‑$95 B
Sources: web_search (https://alphio.ai/blog/nvda-earnings-august-2026)

### 2. Institutional sell pressure (medium)

Multiple large investors have disclosed recent sales of NVDA shares, which could add downward pressure on the stock if the selling continues.

Evidence:
- Osterweis Capital Management Inc. Sells 6,347 Shares
- NVIDIA Corporation $NVDA Stock Sold by Primecap Management Co.
- 268,305 NVIDIA Shares Sold by The Manufacturers Life Insurance Company
Sources: get_news

### 3. Supply‑chain concentration risk (medium)

NVDA’s heavy reliance on TSMC and a few key suppliers makes it vulnerable to geopolitical tensions and fab bottlenecks that could delay deliveries and compress margins.

Evidence:
- NVIDIA Geopolitical Supply Chain Risk Assessment – highlights geopolitical fragmentation and Taiwan concentration
Sources: web_search (https://nova.kapualabs.com/nvda-stock_panel_c/nvidias-geopolitical-supply-chain-a-definitive-risk-assessment/)

## 3. Hedge Strategy Recommendation

**Buy a protective put option** using 3‑month NVDA put, strike ~5% OTM (~$217).

With 30‑day annualised volatility at 29.45% (low) and a 1‑day 95 % VaR of –3.88%, a 5% OTM put caps downside from earnings miss or sell‑off while preserving upside from the current uptrend.

Data points:
- 30‑day annualised volatility: 29.45%
- 1‑day 95 % VaR: –3.88%
Sizing / trigger: Buy 1 contract (100 shares) per 1000 shares held; strike set at $217 (≈5% below current $229.28) with expiration aligned to the earnings date (late Aug)

Sources: get_price_data; calculate_volatility; get_news; llm_sentiment; web_search

*Generated automatically for educational purposes. Not investment advice. Data may be delayed or incomplete, and LLM output can be wrong.*

## 4. Task 3A: error handling
### 4.1 Tool level: errors are returned to the agent as data, never raised
Each error result says what went wrong and what to try instead (only tools that this agent has).

In [11]:
import json
checks = [("get_price_data", {"ticker": "NOT A TICKER"}),
          ("calculate_volatility", {"ticker": TICKER, "window": 2}),
          ("llm_sentiment", {"headlines": []}),
          ("no_such_tool", {})]
for name, args in checks:
    text, status, _ = box.execute("notebook", set(ALL_TOOLS), name, args)
    print(f"{name}({args}) -> {status}: {text[:230]}")

      <- get_price_data [ERROR, 0 ms]: {"status":"error","error":"ToolError: 'NOT A TICKER' is not a valid ticker symbol","suggestion":"check the ticker or retry with a shorter period such as '6mo'"}
get_price_data({'ticker': 'NOT A TICKER'}) -> error: {"status":"error","error":"ToolError: 'NOT A TICKER' is not a valid ticker symbol","suggestion":"check the ticker or retry with a shorter period such as '6mo'"}
      <- calculate_volatility [ERROR, 0 ms]: {"status":"error","error":"invalid arguments: window: Input should be greater than or equal to 5","suggestion":"fix the arguments and call again"}
calculate_volatility({'ticker': 'NVDA', 'window': 2}) -> error: {"status":"error","error":"invalid arguments: window: Input should be greater than or equal to 5","suggestion":"fix the arguments and call again"}
      <- llm_sentiment [ERROR, 0 ms]: {"status":"error","error":"invalid arguments: headlines: List should have at least 1 item after validation, not 0","suggestion":"fix the argument

### 4.2 Agent level: web search is broken, the agent must find another way
`web_search` is replaced by a function that always fails (like a DuckDuckGo rate limit). The task asks the agent to
start with web search, so the failure happens first. The agent reads the error and its suggestion, and replans.

In [12]:
from research_agents.agent import ReActAgent
from research_agents.prompts import RESEARCH_AGENT_SYSTEM

def broken_search(query, max_results):
    raise RuntimeError("202 Ratelimit (simulated DuckDuckGo outage)")

tracer_fail = Tracer(run_id=f"3A-failure-{STAMP}")
failing_agent = ReActAgent("Research Agent", list(ALL_TOOLS), RESEARCH_AGENT_SYSTEM.format(tools=", ".join(ALL_TOOLS)),
                           agent_llm, ToolBox(sentiment_llm=sentiment_llm, tracer=tracer_fail,
                                              search_backend=broken_search), tracer_fail, max_steps=5)
fail_run = failing_agent.run(f"Start with web_search: what are analysts and the news saying about the main risks "
                             f"to {TICKER} this quarter? Give a short answer with sources.")

  [task] Research Agent <- Start with web_search: what are analysts and the news saying about the main risks to NVDA this quarter? Give a short answer with sources.
  [Research Agent] thinks: We need to call web_search with query about NVDA main risks this quarter. Use max_results maybe 5.Reasoning: We need analyst commentary on NVDA's main risks this quarter, so we start with a web search.
  [Research Agent] -> calls web_search({"max_results":5,"query":"NVDA main risks this quarter analyst commentary"})
      <- web_search [ERROR, 0 ms]: {"status":"error","error":"ToolError: web search failed: RuntimeError: 202 Ratelimit (simulated DuckDuckGo outage)","suggestion":"retry ONCE with a shorter, simpler query; or use get_news to get recent headlines instead"}
  [Research Agent] thinks: We got rate limit error. Suggest retry with shorter query. Let's try "NVDA risks analyst".
  [Research Agent] -> calls web_search({"max_results":5,"query":"NVDA risks analyst"})
      <- web_search [ERROR, 

In [13]:
records = pd.DataFrame([{"tool": r.tool, "args": r.args, "status": r.status} for r in fail_run.records])
display(records)
failed = [i for i, r in enumerate(fail_run.records) if r.status in ("error", "empty")]
if failed:
    after = [r.tool for r in fail_run.records[failed[0] + 1:]]
    print(f"Failure at call {failed[0] + 1} ({fail_run.records[failed[0]].tool}). The agent continued with: {after}")
print("\nFinal answer:\n" + fail_run.final_text[:1200])

,tool,args,status
0,web_search,"{'max_results': 5, 'query': 'NVDA main risks this quarter analyst commentary'}",error
1,web_search,"{'max_results': 5, 'query': 'NVDA risks analyst'}",error
2,get_news,"{'n': 15, 'ticker': 'NVDA'}",ok
3,web_search,"{'max_results': 3, 'query': 'NVIDIA risk Q3 2026'}",error
4,web_search,"{'max_results': 3, 'query': 'NVIDIA risk'}",error


Failure at call 1 (web_search). The agent continued with: ['web_search', 'get_news', 'web_search', 'web_search']

Final answer:
**Financial health (as far as we can see from the data)**  
- The recent news flow is dominated by large institutional trades rather than earnings surprises or balance‑sheet alerts.  Market‑beat headlines report that several funds have **sold** sizable blocks of NVDA stock (e.g., 6,347 shares by Osterweis Capital, 268,305 shares by The Manufacturers Life Insurance Company) while a few have **bought** (e.g., 3,087,398 shares by Jones Financial Companies).  This mixed activity suggests that the market still views NVDA as a liquid, high‑profile holding, but the net‑sell pressure from a few large owners hints at a modest‑to‑cautious stance among some investors.  
- No price‑trend or volatility numbers are available in the current tool output, so we cannot quantify recent price performance, SMA cross‑overs, or volatility‑adjusted risk metrics.

**Market sentiment (

## 5. Task 3C: short-term memory
The follow-up runs in the **same session** (LangGraph checkpointer, same thread id). The earlier tool results are in
the conversation, so the agent answers from them. The check below counts tool calls made for the follow-up.

In [14]:
thread = {"configurable": {"thread_id": session.thread_id}}
memory = session.agent.graph.get_state(thread).values["messages"]
print(f"Session memory: {len(memory)} messages, tool results: {[m.name for m in memory if isinstance(m, ToolMessage)]}")

follow = session.ask("Follow-up: what annualised volatility did you calculate, over which window, and what does it "
                     "mean for the hedge you suggested? Use the results you already have.")
print("\nAnswer:", follow.answer)
print("\nTool calls made for the follow-up:", follow.tool_calls)

Session memory: 18 messages, tool results: ['get_price_data', 'calculate_volatility', 'get_news', 'llm_sentiment', 'web_search', 'web_search', 'web_search', 'web_search']

-------- Follow-up (same session): Follow-up: what annualised volatility did you calculate, over which window, and what does it mean for the hedge you suggested? Use the results you already have. --------
  [task] Research Agent <- Follow-up: what annualised volatility did you calculate, over which window, and what does it mean for the hedge you suggested? Use the results you already have.
  [Research Agent] thinks: We need to answer: what annualised volatility did we calculate, over which window, and what does it mean for the hedge suggested. Use existing results: we have calculate_volatility result: annualised_vol_pct 29.45, window 30 days. Need to explain meaning for hedge: low volatility relative to past year, implies put premium relatively cheap, hedge likely effective, etc. Provide concise answer.
  [Research A

## 6. Task 3B: two agents with separate tool access
Each agent is bound only to its own tools (the LLM never sees the others), and `ToolBox.execute()` also refuses
any tool outside the agent's list. Below: what happens if Agent B tries a price tool.

In [15]:
from research_agents.multi_agent import AGENT_A, AGENT_B, TOOLS_A, TOOLS_B, run_research

display(pd.DataFrame([
    {"agent": AGENT_A, "role": "quantitative analysis", "tools": TOOLS_A, "output": "DataBrief (Pydantic)"},
    {"agent": AGENT_B, "role": "qualitative synthesis", "tools": TOOLS_B, "output": "ResearchReport (Pydantic)"}]))

guard = ToolBox(tracer=Tracer(run_id=f"3B-restriction-{STAMP}"))
for agent, tools, tool, args in [(AGENT_B, TOOLS_B, "get_price_data", {"ticker": TICKER}),
                                 (AGENT_A, TOOLS_A, "web_search", {"query": "NVDA news"})]:
    text, status, _ = guard.execute(agent, set(tools), tool, args)
    print(f"{agent} -> {tool}: {status.upper()} {text}")

,agent,role,tools,output
0,Agent A (Data Analyst),quantitative analysis,"[get_price_data, calculate_volatility, llm_sentiment]",DataBrief (Pydantic)
1,Agent B (Research Writer),qualitative synthesis,"[web_search, get_news]",ResearchReport (Pydantic)


      <- get_price_data [DENIED, 0 ms]: {"status":"error","error":"Tool 'get_price_data' is not available to Agent B (Research Writer).","your_tools":["get_news","web_search"],"suggestion":"use only your own tools"}
Agent B (Research Writer) -> get_price_data: DENIED {"status":"error","error":"Tool 'get_price_data' is not available to Agent B (Research Writer).","your_tools":["get_news","web_search"],"suggestion":"use only your own tools"}
      <- web_search [DENIED, 0 ms]: {"status":"error","error":"Tool 'web_search' is not available to Agent A (Data Analyst).","your_tools":["calculate_volatility","get_price_data","llm_sentiment"],"suggestion":"use only your own tools"}
Agent A (Data Analyst) -> web_search: DENIED {"status":"error","error":"Tool 'web_search' is not available to Agent A (Data Analyst).","your_tools":["calculate_volatility","get_price_data","llm_sentiment"],"suggestion":"use only your own tools"}


## 7. Task 3B: the full pipeline (no manual step between query and report)
The trace prints what each agent said, every tool call and result, and every handoff with its schema:
`DataBrief` (A -> B), then the critique loop: `ClarificationRequest` (B -> A) and `ClarificationResponse` (A -> B),
then B's final report. `use_cache=False` forces a fresh run here; section 8 shows the cache.

In [16]:
tracer_3b = Tracer(run_id=f"3B-{STAMP}")
record = run_research(TICKER, llm=agent_llm, json_llm=json_llm, sentiment_llm=sentiment_llm, tracer=tracer_3b,
                      use_cache=False)


-------- Agent A (Data Analyst): quantitative analysis --------
  [task] Agent A (Data Analyst) <- Build a quantitative data brief for NVDA for a 90-day risk review: price trend and indicators, volatility, drawdown and value-at-risk.
  [Agent A (Data Analyst)] thinks: We need to produce a quantitative data brief for NVDA (NVIDIA) for a 90-day risk review: price trend and indicators, volatility, drawdown, VaR. Use tools: get_price_data, calculate_volatility. No news needed. We'll need price trend: maybe SMA-50, SMA-200, RSI, MACD, Bollinger Bands, period return, 52-week range. For 90 days, we can request period "3mo". Use get_price_data with period "3mo". The...
  [Agent A (Data Analyst)] -> calls get_price_data({"period":"3mo","ticker":"NVDA"})
      <- get_price_data [OK, 9 ms]: {"ticker":"NVDA","period":"3mo","start":"2026-07-09","end":"2026-10-09","rows":66,"last_close":229.28,"period_return_pct":13.19,"high_52w":243.37,"low_52w":163.9,"pct_from_52w_high":-5.79,"indicators":{"sma_5

### 7.1 Handoffs (typed messages between the agents)

In [17]:
handoffs = [e for e in tracer_3b.events if e["kind"] == "handoff"]
display(pd.DataFrame([{"from": e["sender"], "to": e["receiver"], "schema": e["schema"],
                       "fields": ", ".join(e["payload"])} for e in handoffs]))
from research_agents.schemas import ClarificationRequest, ClarificationResponse, DataBrief
for e, schema in zip(handoffs, (DataBrief, ClarificationRequest, ClarificationResponse)):
    schema(**e["payload"])                              # each payload validates against its schema
print("All handoff payloads validate against their Pydantic schemas.")

,from,to,schema,fields
0,Agent A (Data Analyst),Agent B (Research Writer),DataBrief,"ticker, as_of, price, volatility, sentiment, analyst_notes, data_gaps, tools_used"
1,Agent B (Research Writer),Agent A (Data Analyst),ClarificationRequest,"question, requested_data, headlines, reason"
2,Agent A (Data Analyst),Agent B (Research Writer),ClarificationResponse,"question, answer, data, tools_used"


All handoff payloads validate against their Pydantic schemas.


### 7.2 Critique loop: request, response, and how the report used it

In [18]:
req, resp = record["clarification"]["request"], record["clarification"]["response"]
print("Agent B asked:", req["question"])
print("Requested data:", req["requested_data"])
print(f"Headlines sent to Agent A: {len(req['headlines'])}")
print("Reason:", req["reason"])
print("\nAgent A answered:", resp["answer"])
print("Agent A used tools:", resp["tools_used"])
print(json.dumps(resp["data"], indent=1)[:1500])
if record["report"]:
    print("\nHow Agent B incorporated it:", record["report"]["clarification_used"])

Agent B asked: What is the aggregated LLM sentiment for the recent NVDA news headlines you provided?
Requested data: ['overall_sentiment_score', 'sentiment_breakdown_by_headline']
Headlines sent to Agent A: 15
Reason: Sentiment from recent news is missing in the brief; it is essential to gauge market mood and complement the technical and volatility analysis for a comprehensive investment recommendation.

Agent A answered: The LLM sentiment analysis of the 15 recent NVDA headlines yields an **overall sentiment score of 0.273 (positive)**, with **9 positive, 1 neutral, and 5 negative** classifications.

**Headline‑by‑headline breakdown**

- “Meet the Only AI Stock I'd Buy Over Nvidia in October” – neutral  
- “Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Corporation” – negative  
- “Palisade Capital Management LP Acquires 28,724 Shares of NVIDIA Corporation” – positive  
- “NVIDIA Corporation Stock Sold by Primecap Management Co.” – negative  
- “Jones Financial Compani

### 7.3 Final report from Agent B

In [19]:
if record["report"]:
    display(Markdown(report_to_markdown(record["report"])))
else:
    print("Report failed:", record["error"])
print("\nTrace summary:", record["trace_summary"])

# NVDA research brief (2026-10-11)

## 1. Financial Health Summary

NVDA is in a short‑term uptrend, trading 13.2% above its 3‑month ago level and staying above both the 50‑day (221.94) and 200‑day (201.83) SMAs, confirming bullish momentum. Valuation is modestly below its 52‑week high (‑5.79%) and technical indicators (RSI 53, MACD positive) are neutral‑to‑bullish. Volatility remains elevated at 38.5% annualised (65th percentile) with a 1‑year max drawdown of 20.2% and a 1‑day 95% VaR of –3.9%, indicating sizable downside risk on adverse moves.

- Price: 229.28
- 3‑mo return: 13.19%
- Annualised 90‑day volatility: 38.51%
- 1‑day 95% VaR: -3.9%
- 1‑yr max drawdown: -20.21%
- 52‑wk high: 243.37 (‑5.79% from current)

**Market sentiment:** LLM sentiment on the latest 15 headlines is positive (score 0.273) with 9 positive, 1 neutral and 5 negative mentions, suggesting a modestly upbeat market mood despite some sell‑side activity.

## 2. Top Three Risks (next 90 days)

### 1. Elevated short‑term volatility and drawdown risk (high)

The stock’s 90‑day annualised volatility of 38.5% and a historical 1‑year peak‑to‑trough decline of 20.2% could trigger sharp price drops within the next 90 days, especially if broader market stress returns.

Evidence:
- Annualised volatility 38.51% (90‑day window)
- Max drawdown 1‑yr -20.21%
- 1‑day 95% VaR -3.9%
Sources: calculate_volatility, get_price_data

### 2. Potential slowdown in AI‑chip demand (medium)

Analyst commentary and recent articles flag a risk that AI data‑center demand may soften later in 2026, which would pressure NVDA’s revenue growth and share price.

Evidence:
- "Nvidia stock forecast 2026 faces a new test as an analyst downgrades NVDA near record highs despite explosive AI growth" (stockminded.com)
- "AI development slowdown: what it means for Nvidia..." (invezz.com)
Sources: web_search

### 3. Intensifying competition from AMD, Intel and custom silicon (medium)

Rising competition in the AI accelerator market could erode NVDA’s market share and pricing power, creating head‑room for a corrective move in the next quarter.

Evidence:
- "The AI Chip Wars: NVIDIA, AMD, and Custom Silicon Explained 2026" (hakia.com)
- "The GPU Market War 2026: NVIDIA vs AMD vs Custom Silicon" (data-gate.ch)
Sources: web_search

## 3. Hedge Strategy Recommendation

**Buy 3‑month (≈90‑day) out‑of‑the‑money put option** using 3‑month put, strike ~218 (≈5% below current 229.28) covering 100 shares.

The hedge caps downside to roughly the 1‑day VaR level (‑3.9%) and protects against the high 38.5% annualised volatility and the 20% historic drawdown, which are the primary risk drivers identified.

Data points:
- Annualised volatility 38.51% (90‑day)
- 1‑day 95% VaR -3.9%
Sizing / trigger: Buy one put contract (100 shares) at strike 218; if price falls below 218 the put offsets losses, otherwise the position expires worthless.

**How the analyst's clarification was used:** Agent A’s sentiment analysis added an overall positive score of 0.273 (9 positive, 5 negative, 1 neutral) which confirmed the modestly upbeat market sentiment reflected in the summary.

Sources: calculate_volatility; get_price_data; get_news; web_search

*Generated automatically for educational purposes. Not investment advice. Data may be delayed or incomplete, and LLM output can be wrong.*


Trace summary: {'run_id': '3B-20261011-105736', 'tool_calls': 8, 'by_tool': {'get_price_data': 1, 'calculate_volatility': 1, 'get_news': 1, 'web_search': 4, 'llm_sentiment': 1}, 'errors': 2, 'empty': 0, 'total_tool_ms': 19339}


## 8. Task 3C: persistent memory
The brief was saved as `memory/<TICKER>_<date>.json`. A second run with the same ticker detects the file and loads it:
no agent and no tool runs (0 tool calls in this run's trace).

In [20]:
tracer_cache = Tracer(run_id=f"3C-cache-{STAMP}")
again = run_research(TICKER, llm=agent_llm, json_llm=json_llm, sentiment_llm=sentiment_llm, tracer=tracer_cache)
print("from_cache:", again["from_cache"], "| file:", again["path"])
print("tool calls in this run:", len(tracer_cache.tool_calls()))
print("same report as the first run:", again["report"] == record["report"])
from pathlib import Path
print("\nmemory/ files:", sorted(p.name for p in Path("memory").glob("*.json")))
print("keys in the saved brief:", list(again))

  [cache_hit] Loaded NVDA_2026-10-11.json from memory; no agents or tools run
from_cache: True | file: /content/CDAZZDEV-MLE-GobishangarVijayendran/task3_agentic/memory/NVDA_2026-10-11.json
tool calls in this run: 0
same report as the first run: True

memory/ files: ['NVDA_2026-10-11.json']
keys in the saved brief: ['ticker', 'date', 'query', 'run_id', 'created_at', 'report', 'data_brief', 'clarification', 'trace_summary', 'error', 'from_cache', 'path']


## 9. Task 3C: observability
Every tool call is one line in `logs/agent_trace.jsonl`: `ts, run_id, agent, tool, inputs, output` (truncated to
200 characters), `duration_ms` (wall clock) and `status`. Agent messages, handoffs and cache events are logged too.

In [21]:
from research_agents.trace_view import load_events, runs, tool_calls

events = load_events(TRACE_FILE)
mine = events[events["run_id"].str.endswith(STAMP)]
print(f"{TRACE_FILE.name}: {len(events)} lines in total, {len(mine)} from this notebook run")
display(runs(mine))
calls_3b = tool_calls(events, f"3B-{STAMP}")
display(calls_3b[["start", "agent", "tool", "inputs", "status", "duration_ms", "output"]])
print("Longest logged output:", int(tool_calls(mine)["output"].str.len().max()), "characters (limit 200)")

agent_trace.jsonl: 73 lines in total, 73 from this notebook run


,run_id,start,end,tool_calls,errors
0,3C-cache-20261011-105736,2026-10-11 11:07:25.425000+00:00,2026-10-11 11:07:25.425000+00:00,0,0
1,3B-20261011-105736,2026-10-11 11:05:13.471000+00:00,2026-10-11 11:07:25.375000+00:00,8,2
2,3B-restriction-20261011-105736,2026-10-11 11:01:43.701000+00:00,2026-10-11 11:01:43.702000+00:00,2,2
3,3A-failure-20261011-105736,2026-10-11 10:59:59.266000+00:00,2026-10-11 11:01:09.630000+00:00,5,4
4,3A-20261011-105736,2026-10-11 10:57:54.007000+00:00,2026-10-11 11:01:43.673000+00:00,8,2
5,tools-20261011-105736,2026-10-11 10:57:45.826000+00:00,2026-10-11 10:59:59.244000+00:00,9,4


,start,agent,tool,inputs,status,duration_ms,output
0,2026-10-11 11:05:14.328000+00:00,Agent A (Data Analyst),get_price_data,"{""period"":""3mo"",""ticker"":""NVDA""}",ok,9.0,"{""ticker"":""NVDA"",""period"":""3mo"",""start"":""2026-07-09"",""end"":""2026-10-09"",""rows"":66,""last_close"":229.28,""period_return_pct"":13.19,""high_52..."
1,2026-10-11 11:05:15.059000+00:00,Agent A (Data Analyst),calculate_volatility,"{""ticker"":""NVDA"",""window"":90}",ok,5.0,"{""ticker"":""NVDA"",""window"":90,""as_of"":""2026-10-09"",""annualised_vol_pct"":38.51,""one_year_vol_pct"":37.66,""vol_percentile_1y"":65.1,""max_draw..."
2,2026-10-11 11:05:17.082000+00:00,Agent B (Research Writer),get_news,"{""n"":15,""ticker"":""NVDA""}",ok,2813.0,"{""ticker"":""NVDA"",""headlines"":[{""title"":""Meet the Only AI Stock I'd Buy Over Nvidia in October"",""publisher"":""fool.com"",""published_at"":""20..."
3,2026-10-11 11:05:20.561000+00:00,Agent B (Research Writer),web_search,"{""max_results"":5,""query"":""NVIDIA analyst downgrade risk 2026 Q4 AI chip demand slowdown""}",ok,3432.0,"{""query"":""NVIDIA analyst downgrade risk 2026 Q4 AI chip demand slowdown"",""results"":[{""title"":""Nvidia Stock Is Booming on AI \u2013 So Wh..."
4,2026-10-11 11:05:37.229000+00:00,Agent B (Research Writer),web_search,"{""max_results"":1,""query"":""Nvidia stock forecast 2026 faces a new test as an analyst downgrades NVDA near record highs despite explosive ...",error,3864.0,"{""status"":""error"",""error"":""ToolError: web search failed: DDGSException: No results found."",""suggestion"":""retry ONCE with a shorter, simp..."
5,2026-10-11 11:05:50.954000+00:00,Agent B (Research Writer),web_search,"{""max_results"":5,""query"":""Nvidia competition AMD Intel AI chip market 2026 risk""}",ok,3973.0,"{""query"":""Nvidia competition AMD Intel AI chip market 2026 risk"",""results"":[{""title"":""The AI Chip Wars: NVIDIA, AMD, and Custom Silicon ..."
6,2026-10-11 11:06:11.850000+00:00,Agent B (Research Writer),web_search,"{""max_results"":5,""query"":""NVIDIA earnings October 2026 date""}",error,4291.0,"{""status"":""error"",""error"":""ToolError: web search failed: DDGSException: No results found."",""suggestion"":""retry ONCE with a shorter, simp..."
7,2026-10-11 11:06:39.856000+00:00,Agent A (Data Analyst),llm_sentiment,"{""headlines"":[""Meet the Only AI Stock I'd Buy Over Nvidia in October"",""Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Co...",ok,952.0,"{""n"":15,""score"":0.273,""label"":""positive"",""counts"":{""positive"":9,""neutral"":1,""negative"":5},""items"":[{""headline"":""Meet the Only AI Stock I..."


Longest logged output: 200 characters (limit 200)


In [22]:
with open(TRACE_FILE, encoding="utf-8") as fh:
    last_call = [line for line in fh if '"kind": "tool_call"' in line][-1]
print(json.dumps(json.loads(last_call), indent=2))

{
  "ts": "2026-10-11T11:06:40.808+00:00",
  "run_id": "3B-20261011-105736",
  "kind": "tool_call",
  "agent": "Agent A (Data Analyst)",
  "tool": "llm_sentiment",
  "inputs": {
    "headlines": [
      "Meet the Only AI Stock I'd Buy Over Nvidia in October",
      "Osterweis Capital Management Inc. Sells 6,347 Shares of NVIDIA Corporation $NVDA - MarketBeat",
      "Palisade Capital Management LP Acquires 28,724 Shares of NVIDIA Corporation $NVDA - MarketBeat",
      "NVIDIA Corporation $NVDA Stock Sold by Primecap Management Co. CA - MarketBeat",
      "Jones Financial Companies Lllp Acquires 3,087,398 Shares of NVIDIA Corporation $NVDA - MarketBeat",
      "268,305 NVIDIA Corporation $NVDA Shares Sold by The Manufacturers Life Insurance Company - MarketBeat",
      "NVIDIA Corporation $NVDA Stock Sold by Kinetic Partners Management LP - MarketBeat",
      "NVIDIA Corporation $NVDA Shares Purchased by Nicholas Investment Partners LP - MarketBeat",
      "NVIDIA Corporation $NVDA Stak

## 10. Tests (offline: fake Yahoo, fake search, scripted models)

In [23]:
!python -m pytest -rs

.............s.............................................              [100%]
=========================== short test summary info ============================
SKIPPED [1] tests/test_dashboard.py:45: could not import 'streamlit.testing.v1': No module named 'streamlit'
58 passed, 1 skipped in 2.05s


## 11. Save the outputs
The reports go to `outputs/`. In Colab, a zip with the trace, the memory file and the reports is downloaded, so they
can be committed to the repository (`task3_agentic/logs/agent_trace.jsonl` is a required deliverable).

In [24]:
import zipfile
from research_agents.config import PACKAGE_ROOT

out = PACKAGE_ROOT / "outputs"
out.mkdir(exist_ok=True)
if result.report:
    (out / f"{TICKER}_3A_single_agent_report.md").write_text(report_to_markdown(result.report), encoding="utf-8")
if record["report"]:
    (out / f"{TICKER}_3B_multi_agent_report.md").write_text(report_to_markdown(record["report"]), encoding="utf-8")

files = [TRACE_FILE, *sorted((PACKAGE_ROOT / "memory").glob("*.json")), *sorted(out.glob("*.md"))]
with zipfile.ZipFile("task3_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f.relative_to(PACKAGE_ROOT))
        print("added", f.relative_to(PACKAGE_ROOT))
if "google.colab" in sys.modules:
    from google.colab import files as colab_files
    colab_files.download("task3_outputs.zip")

added logs/agent_trace.jsonl
added memory/NVDA_2026-10-11.json
added outputs/NVDA_3A_single_agent_report.md
added outputs/NVDA_3B_multi_agent_report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Appendix: all prompts
Prompts are constants in `research_agents/prompts.py` (system = rules, human = data). They are printed here for
transparency, as the citation policy asks.

In [25]:
from research_agents.prompts import all_prompts
for name, text in all_prompts().items():
    print(f"===== {name} =====\n{text}\n")

===== AGENT_RULES =====
How you work:
- You decide which tool to call next, based on what you have observed so far. There is no fixed order.
- Before EVERY tool call, write one short sentence starting with "Reasoning:" that says what you learned from
  the last result and why you need the next tool.
- Read each tool result before deciding the next step. Do not call two tools blindly in a row.
- Memory: if the conversation already contains a tool result that answers a question, use it. Do NOT call the
  same tool again with the same arguments.
- Errors: if a tool returns "status": "error" or an empty result, do not stop. Follow the "suggestion" in the
  result, or try another of your tools to get the same information.
- Use only numbers that appear in tool results. Never invent data.
- Do not repeat near-identical web searches: one or two broad searches with max_results 5 are usually enough.
- When you have enough evidence, stop calling tools and write your final answer in plain text.

